In [2]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 2.3 MB/s eta 0:00:00 MB/s eta 0:00:01:01
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2 0/3 [typing-extensions]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [openai]━━━━━━━━━━━ 2/3 [openai]


In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [9]:
import pandas as pd
import json
df = pd.read_csv("../data/classified_operational_reviews.csv")

df.head()

,userName,content,score,at,thumbsUpCount,review_length,sentiment,rating_sentiment,text_sentiment,rating_text_check,issue_category
0,Dawn Lowe,its hard for older peeps to work on this app,3,2026-06-19 00:29:09,0,44,Neutral,Neutral,Unclear Text,Aligned,UI / Usability
1,Luz Hernandez,sucks,1,2026-06-18 10:20:40,0,5,Negative,Negative,Negative Text,Aligned,UI / Usability
2,Nicole Donovan,Can't upload proof of insurance,1,2026-06-12 01:08:05,0,31,Negative,Negative,Unclear Text,Aligned,UI / Usability
3,Daisaku Kuze,"DOES NOT WORK, won't login. Neither will the m...",1,2026-06-11 20:51:51,0,235,Negative,Negative,Negative Text,Aligned,Payments
4,MARK on point,omg......horrible bright white screen painful ...,1,2026-06-05 04:53:18,2,243,Negative,Negative,Negative Text,Aligned,UI / Usability


In [11]:
def generate_operational_intelligence(issue_category, reviews):
    
    review_text = "\n".join(reviews.tolist())

    prompt = f"""
You are a senior Operations Intelligence Consultant.

You have received customer reviews already classified into the issue category:

{issue_category}

Analyze ONLY these reviews.

Your task:

1. Identify recurring operational themes.
2. Estimate frequency of each theme.
3. Identify probable operational root causes.
4. Explain business impact.
5. Recommend executive actions.
6. Assign priority:
   High
   Medium
   Low

Return ONLY valid JSON.

Format:

{{
 "issue_category":"",

 "themes":[
   {{
      "theme":"",
      "frequency":"",
      "root_cause":"",
      "business_impact":"",
      "recommendation":"",
      "priority":""
   }}
 ]
}}

Reviews:

{review_text}

"""
    response = client.responses.create(
        model="gpt-5.5",
        input=prompt
    )

    return response.output_text

In [12]:
categories = df["issue_category"].unique()

all_results = []
for category in categories:

    reviews = df[df["issue_category"] == category]["content"]

    print(f"Analyzing {category}...")

    result = generate_operational_intelligence(
        category,
        reviews
    )

    all_results.append(result)

Analyzing UI / Usability...
Analyzing Payments...
Analyzing Maintenance...
Analyzing Authentication...
Analyzing Other Operational Issue...
Analyzing Stability...
Analyzing Performance...


In [13]:
with open(
    "../data/operations_intelligence.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(all_results, f, indent=4)